# Monthly To Seasonal Data Conversion

- Converts monthly netCDF data generated by monthly_merged_data_generation.ipynb into seasonal data

- Change the file paths to your file paths at the last code cell.

In [1]:
# import necessary packages
import xarray as xr
import pandas as pd
import numpy as np
import os
import gc

# define seasons for each region
season_dict = {
    'eastern_east_africa':{
        'MAM': [3,4,5],
        'OND': [10,11,12]
    },
    'lake_victoria_basin':{
        'DJF': [12,1,2],
        'MAM': [3,4,5],
        'SON': [9,10,11]
    },
    'west_africa':{
        'JAS': [7,8,9]
    },
    'south_sudan':{
        'MJJ': [5,6,7],
        'JAS': [7,8,9],
        'ASO': [8,9,10]
    },
    'eastern_ukraine':{
        'DJF': [12,1,2],
        'AMJ': [4,5,6],
        'JA': [7,8]
    },
    'southern_africa':{
        'DJF': [12,1,2],
        'FMA': [2,3,4]
    },
    'sri_lanka':{
        'OND': [10,11,12]
    }
}

# Helper function for monthly to seasonal conversion

In [2]:
def convert_monthly_to_seasonal(file_path, seasons_dict, save_path, start_year=1993, end_year=2024):
    '''Takes a file path to a monthly netCDF file (generated by monthly_data_generation.py) and converts it to a seasonal csv file.

    Arguments:
        file_path (str): path to monthly netCDF file
        seasons_dict (dict): dictionary of seasons and months for each region
        save_path (str): path to save seasonal csv file
        start_year (int, optional): start year for seasonal data. Defaults to 1993.
        end_year (int, optional): end year for seasonal data. Defaults to 2024.

    Seasonal Data Columns:
        region - name of region
        model - name of model
        season - season of predictionn
        lead_time - lead time of prediction
        predicted_precip - average predicted precip for the season
        precip - average observed precip for the season
        month_of_prediction - month that the prediction was made
        year_of_prediction - year that the prediction was made
        realization_year - year that the season occurred

    Notes:
        Precip is averaged across all months in the season.
        Predicted precip is averaged across all months in the season for the same lead time and year.
        Handles different file path conventions with os.
        Handles the situation where the season spans multiple years (e.g., DJF).
        Handles the situation where limited predictions are available for a given year. (i.e CDS models were initiated in 1993)
    '''

    # initiate list of dataframes, for regions with multiple seasons
    df_list = []

    # Extract model and region names
    file_name = os.path.basename(file_path)
    name_split = file_name.replace('.nc', '').split('_')
    region_name = '_'.join(name_split[:-2])
    current_model = name_split[-2]
    new_file_name = file_name.replace('.nc', '_seasonal.csv')

    # Status
    print(f'Converting {file_name} to seasonal...')

    # open dataset
    ds = xr.open_dataset(file_path)

    # convert to dataframe, drop nas
    df = ds.to_dataframe().reset_index().dropna()

    # compute spatial and ensemble mean of predicted and actual precip
    df = df.groupby(['time', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()

    # Add time parts
    df['month'] = df['time'].dt.month
    df['year'] = df['time'].dt.year

    # iterate over season and months in seasons_dict
    for season, months in seasons_dict[region_name].items():

        # subset current dataframe to months in the current season
        season_df = df[df['month'].isin(months)].copy()

        # create season, model, and region columns
        season_df['season'] = season
        season_df['model'] = current_model
        season_df['region'] = region_name

        # Filter lead_time
        season_df = season_df[season_df['lead_time'] <= 6.5]

        # Calculate date of prediction

        # Get start month for current season
        start_month = months[0]

        # Floor the lead time before subtraction
        lead_time_floor = np.floor(season_df['lead_time']).astype(int)
        month_pred_int = start_month - lead_time_floor

        # Handle wrap-around
        wrapped_month = month_pred_int.copy()
        wrapped_month[wrapped_month <= 0] += 12

        # Adjust year accordingly
        season_df['year_of_prediction'] = season_df['year'] - (month_pred_int <= 0).astype(int)
        season_df['month_of_prediction'] = wrapped_month

        # Filter by realization year
        season_df = season_df[
            (season_df['year'] >= start_year) &
            (season_df['year'] <= end_year)
        ]

        # Calculate seasonal average precip per season-year using unique values
        seasonal_precip = (
            season_df
            .drop_duplicates(subset=['precip', 'region', 'season', 'year'])
            .groupby(['region', 'season', 'year'])['precip']
            .mean()
            .reset_index()
            .rename(columns={'year': 'realization_year'})
        )

        # Average predicted_precip across all season months for the same lead_time and year
        # ie for MAM, average 3,4,5 predictions made in 3
        predicted_means = (
            season_df
            .groupby(['region', 'season', 'year', 'lead_time', 'month_of_prediction', 'year_of_prediction'])['predicted_precip']
            .mean()
            .reset_index()
            .rename(columns={'year': 'realization_year'})
        )

        # Merge seasonal average precip onto predicted mean precip
        season_df = pd.merge(
            predicted_means,
            seasonal_precip,
            on=['region', 'season', 'realization_year'],
            how='left'
        )

        # Add model name
        season_df['model'] = current_model

        # Final columns
        season_df = season_df[[
            'region', 'model', 'season',
            'lead_time', 'predicted_precip', 'precip',
            'month_of_prediction', 'year_of_prediction', 'realization_year'
        ]]

        # append the season data to the df list
        df_list.append(season_df)

    # create full dataframe for current region and all its seasons
    final_df = pd.concat(df_list, ignore_index=True)

    # Save to save path
    os.makedirs(save_path, exist_ok=True)
    final_df.to_csv(os.path.join(save_path, new_file_name), index=False)

    # Status
    print(f'Successfully Converted {file_name} to seasonal.')
    print(f'Saved to {os.path.join(save_path, new_file_name)}.')
    print('----------------------------------------------------')

# Conversion for all monthly files in monthly path

In [ ]:
'''Example Usage
'''
# file path to monthly netcdf data
monthly_files_path = '/content/drive/My Drive/capstone_data/netCDF'

# save path for seasonal data
save_path = '/content/drive/My Drive/capstone_data/csv'

# iterate over all files in monthly path
for file in os.listdir(monthly_files_path):
    file_path = os.path.join(monthly_files_path, file)

    # Run the conversion
    convert_monthly_to_seasonal(file_path, season_dict, save_path=save_path)

    # Clean up memory
    gc.collect()

Converting west_africa_CMCC_merged.nc to seasonal...
Successfully Converted west_africa_CMCC_merged.nc to seasonal.
Saved to /content/drive/My Drive/capstone_data/csv/west_africa_CMCC_merged_seasonal.csv.
----------------------------------------------------
Converting southern_africa_CMCC_merged.nc to seasonal...
Successfully Converted southern_africa_CMCC_merged.nc to seasonal.
Saved to /content/drive/My Drive/capstone_data/csv/southern_africa_CMCC_merged_seasonal.csv.
----------------------------------------------------
Converting eastern_ukraine_CMCC_merged.nc to seasonal...
Successfully Converted eastern_ukraine_CMCC_merged.nc to seasonal.
Saved to /content/drive/My Drive/capstone_data/csv/eastern_ukraine_CMCC_merged_seasonal.csv.
----------------------------------------------------
Converting eastern_east_africa_CMCC_merged.nc to seasonal...
Successfully Converted eastern_east_africa_CMCC_merged.nc to seasonal.
Saved to /content/drive/My Drive/capstone_data/csv/eastern_east_africa